# 🍎 Apple Products Price Prediction (2020-2026)

**Goal:** Predict the *Current Selling Price (USD)* of Apple products (iPhone, iPad, Mac, Watch) sold on Amazon & Flipkart, using product details, condition, ratings and sale-event info.

**Dataset:** Apple Products Pricing 2020-2026 (80,000 rows)

**Workflow:**
1. Load & explore data
2. Clean data + check for data leakage
3. Feature engineering
4. Train ML model (XGBoost Regressor)
5. Evaluate performance
6. Conclusion


## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

sns.set_style('whitegrid')
%matplotlib inline


## 2️⃣ Load the Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/apple-products-pricing-2020-2026/apple_products_pricing_2020_2026.csv')
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


**Note:** `Sale_Event` has many missing values — this simply means the product was **not on sale** that day. We will fill these with `'None'` instead of dropping rows.

## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
df['Sale_Event'] = df['Sale_Event'].fillna('None')

plt.figure(figsize=(6,4))
df['Product_Category'].value_counts().plot(kind='bar', color='teal')
plt.title('Number of Listings per Product Category')
plt.ylabel('Count')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='Product_Category', y='Current_Price_USD')
plt.title('Current Price Distribution by Category')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(data=df, x='Launch_Price_USD', y='Current_Price_USD', alpha=0.2)
plt.title('Launch Price vs Current Price')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='Condition', y='Current_Price_USD')
plt.title('New vs Renewed/Refurbished Price')
plt.show()


## 4️⃣ Checking for Data Leakage

Two columns look suspicious for predicting `Current_Price_USD`:

- **`Discount_Pct`** → it is *mathematically calculated* directly from `Launch_Price_USD` and `Current_Price_USD`. Using it as a feature would leak the answer.
- **`Current_Price_INR`** → it is just `Current_Price_USD` converted to Rupees (near-perfect correlation). This is the same information in another currency, so it must be dropped too.

Let's confirm this before modeling.

In [ ]:
# correlation check to prove leakage
check = df.copy()
check['calc_discount'] = (check['Launch_Price_USD'] - check['Current_Price_USD']) / check['Launch_Price_USD'] * 100
print("Discount_Pct is derived formula? Max diff:", (check['calc_discount'] - check['Discount_Pct']).abs().max())

print("\nCorrelation Current_Price_USD vs Current_Price_INR:",
      df['Current_Price_USD'].corr(df['Current_Price_INR']))


✅ Confirmed leakage. We will **drop `Discount_Pct` and `Current_Price_INR`** before training. We'll also drop `Launch_Price_INR` since it's just the USD version converted, and `Date` will be converted into `Year`/`Month` instead of used raw.

## 5️⃣ Feature Engineering

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# Encode categorical columns
cat_cols = ['Platform', 'Product_Category', 'Model_Name', 'Condition', 'Sale_Event', 'Stock_Status']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

feature_cols = cat_cols + ['Launch_Price_USD', 'Rating', 'Reviews_Count', 'Year', 'Month']
target_col = 'Current_Price_USD'

X = df[feature_cols]
y = df[target_col]

X.head()


## 6️⃣ Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape, " Test size:", X_test.shape)


## 7️⃣ Train the Model (XGBoost Regressor)

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    random_state=42
)

model.fit(X_train, y_train)


## 8️⃣ Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 Score : {r2:.4f}")
print(f"MAE      : {mae:.2f} USD")
print(f"RMSE     : {rmse:.2f} USD")


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.2, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price (USD)')
plt.ylabel('Predicted Price (USD)')
plt.title('Actual vs Predicted Current Price')
plt.show()


In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(6,4))
importances.plot(kind='bar', color='orange')
plt.title('Feature Importance')
plt.show()
importances


## 9️⃣ Conclusion

- We built a model to predict the **Current Price (USD)** of Apple products using details like launch price, condition, category, platform, ratings, and sale events.
- Two columns (`Discount_Pct` and `Current_Price_INR`) were correctly **dropped due to data leakage**, since they were mathematically derived from the target.
- The **XGBoost Regressor** achieved a strong **R² score of ~0.99**, meaning it explains almost all the variance in current prices.
- As expected, **`Launch_Price_USD`** and **`Condition`** (New vs Renewed) are the most important features — a product's original price and condition largely determine its resale price.
- **Next steps:** try predicting `Discount_Pct` instead (a more business-useful target), or build a time-series forecast of prices across sale events like Black Friday / Big Billion Days.
